# Goldset Question Tagging

Tagging automatique de **toutes les questions** de `goldset_questions_v2` avec des catégories pertinentes pour l'évaluation par ablation du pipeline.

**Modèle** : `gpt-4.1-mini` (rapide, précis, peu coûteux)

**Tags à assigner :**
- `acronym` : contient des acronymes RH (CDD, RIFSEEP, RTT, CMO...)
- `ambiguous` : question vague ou multi-interprétable
- `legal-ref` : mentionne un article de loi, décret, circulaire
- `factual` : question de fait (définition, montant, durée)
- `procedural` : question de procédure (comment faire, démarches)
- `multi-hop` : nécessite de croiser plusieurs sources
- `temporal` : aspect temporel (dates, délais, durées)
- `cross-source` : réponse nécessitant des sources multiples

In [ ]:
import os
import json
import psycopg
from psycopg.rows import dict_row
from openai import OpenAI
import pandas as pd
from collections import Counter

DSN = os.getenv("TUNNEL_DSN") or os.getenv("SCALINGO_POSTGRESQL_URL") or os.getenv("PG_DSN") or os.getenv("DATABASE_URL")
print(f"DSN configured: {bool(DSN)}")
print(f"OPENAI_API_KEY configured: {bool(os.getenv('OPENAI_API_KEY'))}")

## 1. Load ALL questions from goldset_questions_v2

In [ ]:
with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, question, gold_answer, gold_sources, theme, tags, difficulty, goldset_name
            FROM goldset_questions_v2
            ORDER BY id
        """)
        rows = cur.fetchall()

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} questions from goldset_questions_v2")

# Breakdown by goldset
print(f"\nBreakdown by goldset_name:")
for name, count in df['goldset_name'].value_counts().items():
    print(f"  {name}: {count}")

n_common = len([r for r in rows if r['tags'] and 'common_corpus' in r['tags']])
print(f"\n  dont common_corpus (tag): {n_common}")

print(f"\nExisting tags distribution:")
all_tags = [t for tags in df['tags'] if tags for t in tags]
for tag, count in Counter(all_tags).most_common():
    print(f"  {tag}: {count}")
df.head(3)

## 2. Define LLM tagging function

In [ ]:
import re

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
LLM_MODEL = "gpt-4.1-mini"

VALID_TAGS = {"acronym", "ambiguous", "legal-ref", "factual", "procedural", "multi-hop", "temporal", "cross-source"}

TAGGING_SYSTEM = "Tu es un expert en catégorisation de questions RH pour la fonction publique. Tu réponds uniquement en JSON."

TAGGING_PROMPT = """Analyse cette question et attribue-lui les tags pertinents parmi cette liste :

- acronym : la question contient des acronymes RH (ex: CDD, CDI, RIFSEEP, RTT, CMO, CLM, CLD, NBI, GIPA, SEGUR, PPCR, CGFP, ARTT, JRTT, FPE, FPT, FPH, etc.)
- ambiguous : la question est vague, multi-interprétable, ou manque de contexte pour être précise
- legal-ref : la question mentionne explicitement un article de loi, un décret, une circulaire, un arrêté, ou demande une référence juridique
- factual : question de fait direct (définition, montant, taux, durée, nombre)
- procedural : question de procédure (comment faire, quelles démarches, étapes à suivre)
- multi-hop : la réponse nécessite de croiser ou combiner des informations de plusieurs documents/sources différentes
- temporal : la question implique un aspect temporel important (dates, délais, durées, échéances, périodes)
- cross-source : la réponse complète nécessite des informations de sources/éditeurs différents (MATTE + Service-Public par exemple)

RÈGLES :
1. Une question peut avoir PLUSIEURS tags (c'est même courant)
2. Toute question a au minimum 1 tag parmi factual/procedural
3. Sois précis : "acronym" seulement si des sigles/acronymes sont présents dans le texte de la question
4. "ambiguous" = la question pourrait être interprétée de différentes manières
5. "legal-ref" = mentionne explicitement un texte juridique OU demande "quel article...", "quelle loi..."

Question : "{question}"

Réponds avec un JSON : {{"tags": ["tag1", "tag2", ...]}}"""


def tag_question(question: str) -> list[str]:
    """Tag a single question using GPT-4.1-mini with forced JSON output."""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": TAGGING_SYSTEM},
                {"role": "user", "content": TAGGING_PROMPT.format(question=question)},
            ],
            temperature=0.0,
            max_tokens=200,
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content.strip()
        result = json.loads(content)
        tags = result.get("tags", [])
        return [t for t in tags if t in VALID_TAGS]
    except Exception as e:
        print(f"  Error tagging (id): {e} — raw: {content[:200] if 'content' in dir() else 'N/A'}")
        return ["factual"]

print(f"Using model: {LLM_MODEL}")

## 3. Test on a few questions

In [ ]:
# Test on first 5 questions
for _, row in df.head(5).iterrows():
    tags = tag_question(row['question'])
    print(f"Q: {row['question'][:80]}...")
    print(f"  Tags: {tags}")
    print()

## 4. Tag all questions

In [ ]:
import time

results = []
errors = []
total = len(df)
t_start = time.time()

for i, (_, row) in enumerate(df.iterrows()):
    if i % 50 == 0:
        elapsed = time.time() - t_start
        rate = i / elapsed if elapsed > 0 else 0
        eta = (total - i) / rate if rate > 0 else 0
        print(f"Progress: {i}/{total} ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")
    
    tags = tag_question(row['question'])
    results.append({
        'id': row['id'],
        'question': row['question'],
        'existing_tags': row['tags'] or [],
        'new_tags': tags,
    })

t_total = time.time() - t_start
df_results = pd.DataFrame(results)
print(f"\nTagging complete: {len(df_results)} questions tagged in {t_total:.0f}s ({t_total/len(df_results):.2f}s/question)")

## 5. Review tag distribution

In [ ]:
# Distribution of new tags
all_new_tags = [t for tags in df_results['new_tags'] for t in tags]
tag_counts = Counter(all_new_tags)

print("=" * 60)
print("TAG DISTRIBUTION")
print("=" * 60)
for tag, count in tag_counts.most_common():
    pct = 100 * count / len(df_results)
    bar = '█' * int(pct / 2)
    print(f"  {tag:15s} : {count:4d} ({pct:5.1f}%) {bar}")

print(f"\nAvg tags per question: {len(all_new_tags) / len(df_results):.1f}")

# Show some examples per tag
print("\n" + "=" * 60)
print("EXAMPLES PER TAG")
print("=" * 60)
for tag in ["acronym", "ambiguous", "legal-ref", "procedural", "temporal"]:
    examples = df_results[df_results['new_tags'].apply(lambda x: tag in x)].head(3)
    print(f"\n--- {tag} ({tag_counts.get(tag, 0)} questions) ---")
    for _, ex in examples.iterrows():
        print(f"  • {ex['question'][:100]}")

## 6. Update tags in database

Merge new tags with existing tags (keep `common_corpus` and any manual tags, add the new capability tags).

In [ ]:
# Tags that we auto-assign (will be replaced if they already exist)
AUTO_TAGS = {"acronym", "ambiguous", "legal-ref", "factual", "procedural", "multi-hop", "temporal", "cross-source"}

with psycopg.connect(DSN) as conn:
    with conn.cursor() as cur:
        updated = 0
        for _, row in df_results.iterrows():
            existing = set(row['existing_tags']) if row['existing_tags'] else set()
            # Remove old auto-tags, keep manual ones (like common_corpus, red-teaming, etc.)
            manual_tags = existing - AUTO_TAGS
            # Merge: manual tags + new auto tags
            merged = sorted(manual_tags | set(row['new_tags']))
            
            cur.execute(
                "UPDATE goldset_questions_v2 SET tags = %s, updated_at = NOW() WHERE id = %s",
                (merged, row['id'])
            )
            updated += 1
        
        conn.commit()
        print(f"Updated {updated} questions in goldset_questions_v2")

## 7. Verification

In [ ]:
with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, question, tags
            FROM goldset_questions_v2
            ORDER BY id
        """)
        verify_rows = cur.fetchall()

all_tags_post = [t for row in verify_rows for t in (row['tags'] or [])]
tag_counts_post = Counter(all_tags_post)

print(f"Verification: {len(verify_rows)} total questions")
n_common = len([r for r in verify_rows if r['tags'] and 'common_corpus' in r['tags']])
print(f"  dont common_corpus: {n_common}")

print(f"\nFinal tag distribution (all questions):")
for tag, count in tag_counts_post.most_common():
    pct = 100 * count / len(verify_rows)
    print(f"  {tag:15s} : {count:4d} ({pct:5.1f}%)")

# Show questions with each specialty tag
for tag in ["acronym", "ambiguous", "legal-ref"]:
    qs = [r for r in verify_rows if tag in (r['tags'] or [])]
    print(f"\n{tag}: {len(qs)} questions")
    for q in qs[:3]:
        print(f"  • {q['question'][:90]}")